In [1]:
# ==============================================================================
# SUB-OBJECTIVE 1.1: BUSINESS UNDERSTANDING
# ==============================================================================
def display_business_understanding():
    business_context = {
        "Project Title": "Bank Customer Churn Prediction & Retention Analytics",
        "Domain": "Retail Banking / Financial Services",
        "Business Problem": (
            "Customer attrition creates severe revenue loss. Acquiring new banking "
            "customers costs significantly more than retaining existing accounts."
        ),
        "Business Objective": (
            "Build an automated, cloud-monitored DataOps pipeline to analyze customer "
            "profiles, evaluate leading churn indicators, and supply risk insights to "
            "enable targeted retention campaigns before accounts are closed."
        ),
        "Analytical Task": "Supervised Binary Classification (Target: 'Exited' [0 = Retained, 1 = Churned])",
        "Dataset Volume": "10,000 Records (Meets the minimum 5,000 record constraint)"
    }

    print("=" * 70)
    print("SUB-OBJECTIVE 1.1: BUSINESS UNDERSTANDING")
    print("=" * 70)
    for key, value in business_context.items():
        print(f"[{key}]:\n  {value}\n")
    print("=" * 70)

display_business_understanding()

SUB-OBJECTIVE 1.1: BUSINESS UNDERSTANDING
[Project Title]:
  Bank Customer Churn Prediction & Retention Analytics

[Domain]:
  Retail Banking / Financial Services

[Business Problem]:
  Customer attrition creates severe revenue loss. Acquiring new banking customers costs significantly more than retaining existing accounts.

[Business Objective]:
  Build an automated, cloud-monitored DataOps pipeline to analyze customer profiles, evaluate leading churn indicators, and supply risk insights to enable targeted retention campaigns before accounts are closed.

[Analytical Task]:
  Supervised Binary Classification (Target: 'Exited' [0 = Retained, 1 = Churned])

[Dataset Volume]:
  10,000 Records (Meets the minimum 5,000 record constraint)



In [2]:
# ==============================================================================
# INSTALLATION & PREFECT CLOUD AUTHENTICATION
# ==============================================================================
!pip install -q prefect pandas scikit-learn seaborn matplotlib requests

import os

# Authenticate with your Prefect Cloud API key
os.environ["PREFECT_API_KEY"] = "pnu_lakyzonjgZPJqNjsonUsN15wp2xzVw3q2Rk0"
!prefect cloud login -k $PREFECT_API_KEY

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [3]:
# ==============================================================================
# SUB-OBJECTIVES 1.2 to 1.5: PIPELINE EXECUTION, LOGGING & SCHEDULING
# ==============================================================================
import time
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from prefect import task, flow
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

@task
def ingest_data():
    """Sub-Objective 1.2: Ingest public dataset with >5k records"""
    url = "https://raw.githubusercontent.com/hamzanasirr/Exploratory-Data-Analysis-on-Bank-Customer-Churn-data/master/Churn_Modelling.csv"
    df = pd.read_csv(url)
    print(f"Data Ingested Successfully: {df.shape[0]} rows, {df.shape[1]} columns")
    return df

@task
def preprocess_data(df: pd.DataFrame):
    """Sub-Objective 1.3: Pre-processing (summary stats, missing values, dtypes, scaling)"""
    print("--- 1. Summary Statistics ---")
    print(df.describe())
    print("\n--- 2. Data Types ---")
    print(df.dtypes)

    # Drop non-predictive identifier columns
    df_clean = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

    # Impute missing values for numeric columns
    numeric_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns
    imputer = SimpleImputer(strategy='mean')
    df_clean[numeric_cols] = imputer.fit_transform(df_clean[numeric_cols])

    # Normalize numeric features (excluding target variable 'Exited')
    features_to_scale = [col for col in numeric_cols if col != 'Exited']
    scaler = StandardScaler()
    df_clean[features_to_scale] = scaler.fit_transform(df_clean[features_to_scale])
    print("\nPre-processing complete: Imputation and standard scaling applied.")
    return df_clean

@task
def exploratory_data_analysis(df: pd.DataFrame):
    """Sub-Objective 1.4: EDA (encoding, binning, correlation, plotting, feature importance)"""
    df_eda = df.copy()

    # 1. Categorical feature encoding
    le = LabelEncoder()
    for cat_col in df_eda.select_dtypes(include=['object']).columns:
        df_eda[cat_col] = le.fit_transform(df_eda[cat_col])

    # 2. Binning age into quartiles
    df_eda['Age_Bin'] = pd.qcut(df_eda['Age'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    print("EDA: Age feature binned into 4 quartiles.")

    # 3. Correlation matrix computation and visualization
    numeric_df = df_eda.select_dtypes(include=['float64', 'int64'])
    corr = numeric_df.corr()
    print("\nCorrelation with Target (Exited):\n", corr['Exited'].sort_values(ascending=False))

    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=False, cmap="coolwarm")
    plt.title("Bank Churn Feature Correlation Matrix")
    plt.tight_layout()
    plt.savefig("correlation_heatmap.png")
    plt.close()
    print("EDA: Correlation heatmap saved as 'correlation_heatmap.png'.")

    # 4. Feature importance assessment
    X = numeric_df.drop(columns=['Exited'])
    y = numeric_df['Exited'].astype(int)
    rf = RandomForestClassifier(n_estimators=50, random_state=42)
    rf.fit(X, y)

    importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("\n--- Feature Importance Ranking ---\n", importance)

@flow(name="Bank-Churn-DataOps-Pipeline", log_prints=True)
def ml_pipeline():
    raw_data = ingest_data()
    cleaned_data = preprocess_data(raw_data)
    exploratory_data_analysis(cleaned_data)

# Sub-Objective 1.5: Scheduled 2-minute recurring loop (3 cycles)
for i in range(3):
    print(f"\n================ Running Cycle #{i+1} ================")
    ml_pipeline()
    if i < 2:
        print("\nWaiting 120 seconds (2 minutes) for next scheduled execution...")
        time.sleep(120)


================ Running Cycle #1 ================


INFO:prefect.flow_runs:Beginning flow run 'large-snake' for flow 'Bank-Churn-DataOps-Pipeline'
INFO:prefect.flow_runs:View at https://app.prefect.cloud/account/a5203e8a-655d-428d-8b3c-ea8fa7d34cc1/workspace/f8777847-29a4-485b-9afe-70a9ccb921fc/runs/flow-run/06aa2f93-7077-70c3-8000-9cf60b0307f3
INFO:prefect.task_runs:Data Ingested Successfully: 10000 rows, 14 columns
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:--- 1. Summary Statistics ---
INFO:prefect.task_runs:         RowNumber    CustomerId   CreditScore           Age        Tenure  \
count  10000.00000  1.000000e+04  10000.000000  10000.000000  10000.000000   
mean    5000.50000  1.569094e+07    650.528800     38.921800      5.012800   
std     2886.89568  7.193619e+04     96.653299     10.487806      2.892174   
min        1.00000  1.556570e+07    350.000000     18.000000      0.000000   
25%     2500.75000  1.562853e+07    584.000000     32.000000      3.000000   
50%     5000.50000  1.569074e+07  


Waiting 120 seconds (2 minutes) for next scheduled execution...

================ Running Cycle #2 ================


INFO:prefect.flow_runs:Beginning flow run 'curly-hamster' for flow 'Bank-Churn-DataOps-Pipeline'
INFO:prefect.flow_runs:View at https://app.prefect.cloud/account/a5203e8a-655d-428d-8b3c-ea8fa7d34cc1/workspace/f8777847-29a4-485b-9afe-70a9ccb921fc/runs/flow-run/06aa2f9b-209a-799b-8000-56c7d8cca0f7
INFO:prefect.task_runs:Data Ingested Successfully: 10000 rows, 14 columns
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:--- 1. Summary Statistics ---
INFO:prefect.task_runs:         RowNumber    CustomerId   CreditScore           Age        Tenure  \
count  10000.00000  1.000000e+04  10000.000000  10000.000000  10000.000000   
mean    5000.50000  1.569094e+07    650.528800     38.921800      5.012800   
std     2886.89568  7.193619e+04     96.653299     10.487806      2.892174   
min        1.00000  1.556570e+07    350.000000     18.000000      0.000000   
25%     2500.75000  1.562853e+07    584.000000     32.000000      3.000000   
50%     5000.50000  1.569074e+07


Waiting 120 seconds (2 minutes) for next scheduled execution...

================ Running Cycle #3 ================


INFO:prefect.flow_runs:Beginning flow run 'athletic-gorilla' for flow 'Bank-Churn-DataOps-Pipeline'
INFO:prefect.flow_runs:View at https://app.prefect.cloud/account/a5203e8a-655d-428d-8b3c-ea8fa7d34cc1/workspace/f8777847-29a4-485b-9afe-70a9ccb921fc/runs/flow-run/06aa2fa2-d17f-7081-8000-64d26eeabb5e
INFO:prefect.task_runs:Data Ingested Successfully: 10000 rows, 14 columns
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:--- 1. Summary Statistics ---
INFO:prefect.task_runs:         RowNumber    CustomerId   CreditScore           Age        Tenure  \
count  10000.00000  1.000000e+04  10000.000000  10000.000000  10000.000000   
mean    5000.50000  1.569094e+07    650.528800     38.921800      5.012800   
std     2886.89568  7.193619e+04     96.653299     10.487806      2.892174   
min        1.00000  1.556570e+07    350.000000     18.000000      0.000000   
25%     2500.75000  1.562853e+07    584.000000     32.000000      3.000000   
50%     5000.50000  1.569074e

In [4]:
# ==============================================================================
# SUB-OBJECTIVE 2: API DATA RETRIEVAL & HTTP STATUS VERIFICATION
# ==============================================================================
import requests

API_KEY = "pnu_lakyzonjgZPJqNjsonUsN15wp2xzVw3q2Rk0"
BASE_URL = "https://api.prefect.cloud/api/accounts/a5203e8a-655d-428d-8b3c-ea8fa7d34cc1/workspaces/f8777847-29a4-485b-9afe-70a9ccb921fc"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

print("=" * 70)
print("SUB-OBJECTIVE 2: BUILT-IN API DETAILS & STATUS VERIFICATION")
print("=" * 70)

# Detail 1: Flow Metadata
res_flows = requests.post(f"{BASE_URL}/flows/filter", headers=HEADERS, json={})
print(f"1. Flow Metadata API Status: HTTP {res_flows.status_code}")
if res_flows.json():
    flow_info = res_flows.json()[0]
    print(f"   - Flow Name: {flow_info.get('name')}")
    print(f"   - Flow ID: {flow_info.get('id')}\n")

# Detail 2: Flow Runs & Scheduled Run Details
res_runs = requests.post(f"{BASE_URL}/flow_runs/filter", headers=HEADERS, json={"limit": 3})
print(f"2. Flow Runs API Status: HTTP {res_runs.status_code}")
for idx, run in enumerate(res_runs.json(), 1):
    print(f"   - Run #{idx}: {run.get('name')} | State: {run.get('state_name')} | Duration: {run.get('total_run_time', 0):.2f}s")

# Detail 3: Task Run Details
res_tasks = requests.post(f"{BASE_URL}/task_runs/filter", headers=HEADERS, json={"limit": 3})
print(f"\n3. Task Runs API Status: HTTP {res_tasks.status_code}")
for task in res_tasks.json():
    print(f"   - Task: {task.get('name')} | State: {task.get('state_name')}")

# Detail 4: Pipeline Activity Logs
res_logs = requests.post(f"{BASE_URL}/logs/filter", headers=HEADERS, json={"limit": 3})
print(f"\n4. Activity Logs API Status: HTTP {res_logs.status_code}")
for log in res_logs.json():
    print(f"   - [{log.get('level')}] {log.get('message')}")
print("=" * 70)

SUB-OBJECTIVE 2: BUILT-IN API DETAILS & STATUS VERIFICATION
1. Flow Metadata API Status: HTTP 200
   - Flow Name: Bank-Churn-DataOps-Pipeline
   - Flow ID: e9706850-f585-4841-a4cf-519d34004f49

2. Flow Runs API Status: HTTP 200
   - Run #1: athletic-gorilla | State: Completed | Duration: 2.19s
   - Run #2: curly-hamster | State: Completed | Duration: 1.96s
   - Run #3: large-snake | State: Completed | Duration: 2.24s

3. Task Runs API Status: HTTP 200
   - Task: exploratory_data_analysis-6f0 | State: Completed
   - Task: preprocess_data-a7f | State: Completed
   - Task: ingest_data-299 | State: Completed

4. Activity Logs API Status: HTTP 200
   - [20] Beginning flow run 'copper-cuscus' for flow 'Bank-Churn-DataOps-Pipeline'
   - [20] Data Ingested Successfully: 10000 rows, 14 columns
   - [20] Finished in state Completed()
